# BONUS · B1 · How Data Gets Into the Lakehouse

> **This module is optional.** You can work through it independently after the course or together with the trainer if time allows. It does not depend on any previous module completing successfully — it builds its own demo data and cleans up after itself.

## Learning Objectives

By the end of this notebook you can:

- explain why Volumes are the recommended landing zone for files in Unity Catalog,
- read a CSV file into a Spark DataFrame using both `inferSchema` and an explicit schema,
- query a file directly in SQL using `read_files()`,
- create a Delta table from a query using CTAS (`CREATE TABLE AS SELECT`),
- describe what COPY INTO and Auto Loader do and when to use each,
- explain the ingestion method comparison: CTAS vs COPY INTO vs Auto Loader vs Lakeflow Connect.

## Business Context

You have been working with Gold tables throughout this course. Those tables were built by data engineers who loaded raw files into the Lakehouse. This notebook shows that journey: from a CSV file landing in a Databricks Volume to a queryable Delta table ready for reporting.

## Setup

In [ ]:
%run ../../setup/00_setup

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
    ("DATASET_PATH", DATASET_PATH),
], ["Variable", "Value"]))

In [ ]:
required = [f"{SILVER}.customers"]
missing = [t for t in required if not spark.catalog.tableExists(t)]
if missing:
    raise Exception(f"Missing tables: {missing}. Run data/generate_training_dataset.ipynb first.")
print("[OK] Pre-check passed")

## Unity Catalog Volumes: the file landing zone

Before data becomes a Delta table, raw files need a place to live. In Databricks, that place is a **Unity Catalog Volume** — a managed, governed storage location.

| Question | Answer |
|---|---|
| Where do files land? | `/Volumes/<catalog>/<schema>/<volume>/` |
| Who governs access? | Unity Catalog (same permissions model as tables) |
| What can you store? | Any file format: CSV, JSON, Parquet, images, PDFs |
| How do you read them? | `spark.read`, `read_files()` in SQL, or Autoloader |

The Volume for this course is at `DATASET_PATH` = `/Volumes/{CATALOG}/default/datasets`. It was created by the trainer setup notebook. We will write a small demo CSV there and then read it back using different methods.

## Step 1 — Write a demo CSV to the Volume

In a real project, CSV files arrive from source systems (ERP exports, CRM dumps, API outputs). Here we simulate that by writing a subset of our Silver customer table as CSV.

In [ ]:
DEMO_CSV_PATH = f"{DATASET_PATH}/bonus_b1_demo_customers"

# Write 200 customers from Silver as a single CSV file
(
    spark.sql(f"""
        SELECT customer_id, customer_name, email, city, region, segment, created_date
        FROM {SILVER}.customers
        LIMIT 200
    """)
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(DEMO_CSV_PATH)
)

files = dbutils.fs.ls(DEMO_CSV_PATH)
csv_files = [f for f in files if f.name.endswith(".csv")]
print(f"[OK] Written to Volume: {DEMO_CSV_PATH}")
for f in csv_files:
    print(f"  {f.name}  ({f.size:,} bytes)")

DEMO_CSV_FILE = csv_files[0].path
print("\nReading back from:", DEMO_CSV_FILE)

## Step 2 — Read with `spark.read` (inferSchema)

`spark.read.format("csv")` is the standard PySpark reader for flat files. With `inferSchema=true`, Spark scans the file twice: once to detect column types, once to load data.

> **Production note:** `inferSchema` is convenient for exploration but expensive on large files and can produce unstable types across runs. Always use an explicit schema in pipelines.

In [ ]:
df_inferred = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(DEMO_CSV_FILE)
)

print("Inferred schema:")
df_inferred.printSchema()
display(df_inferred.limit(5))

## Step 3 — Read with an explicit schema

An explicit `StructType` guarantees column types regardless of file content and eliminates the second scan. This is the recommended approach for production pipelines.

| Option | `inferSchema=true` | Explicit `StructType` |
|---|---|---|
| Scans | 2 (detect types + load) | 1 (load only) |
| Type stability | depends on file sample | guaranteed |
| Production suitability | exploration only | recommended |

In [ ]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, DateType

customer_schema = StructType([
    StructField("customer_id",   LongType(),   nullable=False),
    StructField("customer_name", StringType(), nullable=True),
    StructField("email",         StringType(), nullable=True),
    StructField("city",          StringType(), nullable=True),
    StructField("region",        StringType(), nullable=True),
    StructField("segment",       StringType(), nullable=True),
    StructField("created_date",  DateType(),   nullable=True),
])

df_explicit = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(customer_schema)
    .load(DEMO_CSV_FILE)
)

print("Explicit schema:")
df_explicit.printSchema()
display(df_explicit.limit(5))
print(f"\nRow count: {df_explicit.count()}")

## Step 4 — SQL-native file reading with `read_files()`

Databricks Runtime 13.3+ includes `read_files()` — a SQL function that reads files from Unity Catalog Volumes directly in SQL queries. It is the recommended approach for SQL-first workflows.

```sql
SELECT * FROM read_files('/Volumes/catalog/schema/volume/file.csv', format => 'csv', header => true)
```

`read_files()` also injects a `_metadata` struct with file-level information — useful for lineage tracking.

In [ ]:
# Read the same CSV file using read_files() in SQL
display(spark.sql(f"""
    SELECT
        customer_id,
        customer_name,
        city,
        region,
        segment,
        _metadata.file_name  AS source_file
    FROM read_files(
        '{DEMO_CSV_FILE}',
        format  => 'csv',
        header  => true
    )
    LIMIT 10
"""))

## Step 5 — CTAS: Create Table As Select

CTAS is the simplest way to persist a query result as a Delta table. One SQL statement reads the source and writes the Delta table.

```sql
CREATE OR REPLACE TABLE target AS
SELECT ... FROM source
```

**Key limitation:** CTAS does not track which files were already loaded. Every re-run overwrites the table from scratch — no incremental processing, no file tracking. For one-time loads and prototyping it is ideal; for recurring data arrivals use COPY INTO or Auto Loader instead.

In [ ]:
CTAS_TABLE = f"{BRONZE}.bonus_b1_customers_ctas"

spark.sql(f"""
    CREATE OR REPLACE TABLE {CTAS_TABLE}
    COMMENT 'Bonus demo: customers loaded via CTAS from Volume CSV'
    AS
    SELECT
        customer_id,
        customer_name,
        city,
        region,
        segment,
        CAST(created_date AS DATE) AS created_date,
        current_timestamp()        AS _ingested_at,
        '{DEMO_CSV_FILE}'          AS _source_file
    FROM read_files('{DEMO_CSV_FILE}', format => 'csv', header => true)
""")

result = spark.sql(f"SELECT COUNT(*) AS rows, MIN(created_date) AS min_date, MAX(created_date) AS max_date FROM {CTAS_TABLE}")
display(result)
print(f"[OK] Table created: {CTAS_TABLE}")

In [ ]:
# CTAS is NOT incremental — re-running it overwrites the entire table
# Demonstrate: row count stays the same after a second run (no double-loading)
before = spark.table(CTAS_TABLE).count()

spark.sql(f"""
    CREATE OR REPLACE TABLE {CTAS_TABLE}
    AS SELECT *, current_timestamp() AS _ingested_at, '{DEMO_CSV_FILE}' AS _source_file
    FROM read_files('{DEMO_CSV_FILE}', format => 'csv', header => true)
""")

after = spark.table(CTAS_TABLE).count()

display(spark.createDataFrame([
    ("Before re-run", before),
    ("After re-run",  after),
    ("Difference",    after - before),
], ["State", "Rows"]))
print("CTAS = full overwrite. No file tracking. Row count stays the same.")

## COPY INTO — Idempotent Batch Ingestion

COPY INTO adds file tracking on top of CTAS: it records which files have already been loaded into the target table and skips them on subsequent runs. This makes it safe to schedule as a recurring batch job.

```sql
COPY INTO bronze.customers
FROM '/Volumes/catalog/schema/volume/customers/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
```

| Feature | CTAS | COPY INTO |
|---|---|---|
| Incremental | No | Yes |
| File tracking | No | Yes (metadata table) |
| Idempotent | No | Yes |
| Schema evolution | No | Limited (`mergeSchema`) |
| Scalability | Low | Medium (up to ~100K files) |
| Best for | One-time loads | Scheduled batch from cloud storage |

> **When to choose COPY INTO:** scheduled daily/hourly batch loads where new files arrive in a known path and the total number of files is manageable.

## Auto Loader — Scalable Streaming Ingestion

Auto Loader (`cloudFiles`) is Databricks's recommended ingestion method for new projects. It uses file notifications (event-based) or file listing to detect new files as they arrive, processes each file exactly once, and can handle millions of files at scale.

```python
# Auto Loader reads from a Volume path and writes to a Delta table
(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{DATASET_PATH}/schemas/customers")
    .load(f"{DATASET_PATH}/customers/")
    .writeStream
    .format("delta")
    .option("checkpointLocation", f"{DATASET_PATH}/checkpoints/customers")
    .trigger(availableNow=True)        # process all pending files then stop
    .toTable(f"{BRONZE}.customers")
    .awaitTermination()
)
```

| Feature | COPY INTO | Auto Loader |
|---|---|---|
| Incremental | Yes | Yes |
| File tracking | Metadata table | Checkpoint + notifications |
| Scalability | Medium | High (millions of files) |
| Schema evolution | Limited | Advanced (rescue column, hints) |
| Streaming | No | Yes |
| Trigger modes | Scheduled | `availableNow`, `once`, continuous |

> **When to choose Auto Loader:** high-volume pipelines where new files arrive continuously, or when COPY INTO starts slowing down due to the number of files tracked.

## Lakeflow Connect — Zero-Code SaaS Ingestion

For data that lives in SaaS applications (Salesforce, Workday, SAP) rather than files, Databricks provides **Lakeflow Connect** — a managed, no-code connector service.

1. Configure in **Data Engineering → Lakeflow Connect** UI
2. Authenticate with the SaaS application
3. Select objects to ingest
4. Databricks creates a managed pipeline that loads data incrementally into Bronze Delta tables

| Method | Source | Code Required | Best For |
|---|---|---|---|
| CTAS | Files in Volumes | SQL | One-time loads, prototyping |
| COPY INTO | Files in Volumes | SQL | Scheduled batch (< 100K files) |
| Auto Loader | Files in Volumes | Python | High-volume, continuous arrivals |
| Lakeflow Connect | SaaS apps, databases | None (UI) | CRM, ERP, HR system integration |
| JDBC | Relational databases | Python | Custom DB connectivity |

## Summary

| Concept | Key takeaway |
|---|---|
| Unity Catalog Volumes | standard landing zone for files before ingestion |
| `spark.read.csv()` | Python API for file reading; prefer explicit schema in production |
| `read_files()` | SQL-native Volume reader (DBR 13.3+); adds `_metadata` lineage |
| CTAS | simplest method — full overwrite, no file tracking |
| COPY INTO | idempotent batch — tracks files, skips already-loaded ones |
| Auto Loader | streaming ingestion — scales to millions of files |
| Lakeflow Connect | zero-code SaaS/DB connectors via UI |

The Gold tables you queried throughout this course were built using patterns like COPY INTO and Auto Loader feeding Bronze, then Silver transformations, then Gold aggregations. **Bonus B2** shows that full pipeline end-to-end.

## Cleanup

Drop the demo table and Volume files created by this notebook.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {BRONZE}.bonus_b1_customers_ctas")
print("[OK] Dropped table: bonus_b1_customers_ctas")

try:
    dbutils.fs.rm(DEMO_CSV_PATH, recurse=True)
    print(f"[OK] Removed Volume files: {DEMO_CSV_PATH}")
except Exception as e:
    print(f"[INFO] Could not remove Volume files: {e}")